In [ ]:
from IPython.display import display, Markdown
import re
import pandas as pd
import itertools
from tqdm import tqdm
import itertools
import datetime

from openai import OpenAI

In [ ]:
articles_df = pd.read_excel('pubmed_results_549.xlsx')
articles_df.head()

In [ ]:
len(articles_df)

In [ ]:
BATCH_SIZE = 10

In [ ]:
SYSTEM_PROMPT = '''
You are a scientist. You are given a set of inclusion and exclusion criteria and a list of articles. You must decide
whether to include each article and explain why. The format should be as follows:

```
## ARTICLE 1

<reasoning>

INCLUDE or EXCLUDE
```

You must make a decision for **each** input.
'''

In [ ]:
CRITERIA = '''
## INCLUSION CRITERIA

- Study type: Observational studies published in peer-reviewed literature.
- Participants: Individuals tested for SARS-CoV-2 by serological or molecular tests, with or without control groups.
- Outcomes: Studies reporting physical and/or mental health symptoms, symptom clusters, new onset symptoms, or ongoing symptoms following initial SARS-CoV-2 infection.
- Timeframe: No restriction on the time elapsed between infection and symptom reporting, as long as participants had a prior SARS-CoV-2 infection.
- Language: Only studies published in English.
- Publication date: Studies published between January 2020 and August 2022.
- Geographic location: No restrictions applied.
'''

In [ ]:
def format_article(index, article):
    if isinstance(article.publication_date, datetime.datetime):
        pub_date = article.publication_date.date()
    else:
        pub_date = article.publication_date
    
    return f'''
## ARTICLE {index + 1}

TITLE: {article.title}
AUTHORS: {article.authors}
PUBLISHED ON: {pub_date}
ABSTRACT:
{article.abstract}
'''

def make_prompt(batch):
    parts = []
    for i, item in enumerate(batch):
        parts.append(format_article(i, item))
    parts.append(CRITERIA)
    return '\n'.join(parts)

In [ ]:
# Should be present in itertools
def batched(iterable, n):
    # batched('ABCDEFG', 3) → ABC DEF G
    if n < 1:
        raise ValueError('n must be at least one')
    iterator = iter(iterable)
    while batch := tuple(itertools.islice(iterator, n)):
        yield batch

In [ ]:
client = OpenAI(
    # This is the default and can be omitted
    api_key='YOUR_API_KEY',
)
MODEL = 'gpt-4o'

In [ ]:
batch = next(iter(batched(articles_df.itertuples(), 10)))
parts = []
for i, item in enumerate(batch):
    parts.append(format_article(i, item))
parts.append(CRITERIA)
prompt = '\n'.join(parts)


In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    input=prompt)

In [ ]:
def parse_output(output):
    decisions = []
    for line in output.splitlines():
        if match := re.search(r'ARTICLE (\d+)\s*$', line):
            article_ind = int(match.group(1))
            if article_ind != len(decisions) + 1:
                raise ValueError(f'Wrong article: line "{line}", decisions so far {decisions}')
        elif match := re.search(r'\b(IN|EX)CLUDE\b', line):
            if match.group() == 'INCLUDE':
                decisions.append(True)
            elif match.group() == 'EXCLUDE':
                decisions.append(False)
            else:
                raise ValueError(f'I don\'t understand this decision: {line}')
    return decisions

In [ ]:
def query_model(prompt):
    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=prompt)
    return response.output_text

In [ ]:
batches = list(batched(articles_df.itertuples(), BATCH_SIZE))
prompts = [make_prompt(batch) for batch in batches]
outputs = []

In [ ]:
for prompt in tqdm(list(prompts)):
    outputs.append(query_model(prompt))

In [ ]:
all_decisions = []
for batch, output in zip(batches, outputs):
    batch_decisions = parse_output(output)
    if len(batch_decisions) != len(batch):
        print('BAD')
    all_decisions += batch_decisions

assert len(all_decisions) == len(articles_df)
sum(all_decisions), len(all_decisions)

In [ ]:
for output in outputs:
    print (output)

In [ ]:
articles_df['include'] = all_decisions

In [ ]:
articles_df.to_excel('pubmed_results_with_decisions.xlsx', index=False)